In [2]:
from langchain_groq import ChatGroq
import os
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import yfinance as yf
from langchain.agents import AgentType, initialize_agent
from langchain_community.tools.yahoo_finance_news import YahooFinanceNewsTool
from langchain_groq import ChatGroq

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

from pymongo.operations import SearchIndexModel
from langchain_community.vectorstores import AtlasDB
# from langchain_mongodb import MongoDBAtlasVectorSearch
# from pymongo import MongoClient
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser
import PyPDF2
import pandas as pd
from langchain_core.documents import Document
from typing import Iterable
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter
)
from langchain.text_splitter import NLTKTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.utils import ConfigurableFieldSpec
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

/home/steeldev/Desktop/Github/PotatoTrade/mlenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import feedparser
from urllib.parse import quote
from typing import List, Dict, Optional


class GoogleNews:
    """
    A class to search Google News.
    """

    def __init__(self):
        """
        Initializes the base URL.
        """
        self.base_url = "https://news.google.com/rss"

    def _fetch_news(self, url: str, k: int = 3) -> List[Dict[str, str]]:
        """
        Fetches news from the given URL.
        """
        news_data = feedparser.parse(url)
        return [
            {"title": entry.title, "link": entry.link}
            for entry in news_data.entries[:k]
        ]

    def _collect_news(self, news_list: List[Dict[str, str]]) -> List[Dict[str, str]]:
        """
        Organizes and returns the news list.
        """
        if not news_list:
            print("No news found for the given keyword.")
            return []

        result = []
        for news in news_list:
            result.append({"url": news["link"], "content": news["title"]})

        return result

    def search_latest(self, k: int = 3) -> List[Dict[str, str]]:
        """
        Searches for the latest news.
        """
        url = f"{self.base_url}?hl=en&gl=US&ceid=US:en"
        news_list = self._fetch_news(url, k)
        return self._collect_news(news_list)

    def search_by_keyword(
        self, keyword: Optional[str] = None, k: int = 3
    ) -> List[Dict[str, str]]:
        """
        Searches for news by keyword.
        """
        if keyword:
            encoded_keyword = quote(keyword)
            url = f"{self.base_url}/search?q={encoded_keyword}&hl=en&gl=US&ceid=US:en"
        else:
            url = f"{self.base_url}?hl=en&gl=US&ceid=US:en"
        news_list = self._fetch_news(url, k)
        return self._collect_news(news_list)

In [5]:
news_tool = GoogleNews()
news_tool.search_by_keyword("Microsoft", 3)

[{'url': 'https://news.google.com/rss/articles/CBMisAFBVV95cUxNeVNyZFQ5SEJvMTNZY2RDeHQ2bDNVY21BR2dJbDVDNzdmSEpET0pxOVlWRS05VVBhX3dlbE9lVk5NU1daVDRNYmNUMUdHcWNJQUhvOGRibm5ucWp0MlJlMnpSU3RDOVoxQ20zaEhHVFl5RFhwbHlmMzBoaG1tbmFCTDFjWWVkWVczX3VuZWFhcGZGdFNoLVNvd0t4X0NUYWlfck4tTjN6enpna25Lbkkwag?oc=5',
  'content': 'Microsoft Warns 1 Billion Windows Users—Do Not Use Password - Forbes'},
 {'url': 'https://news.google.com/rss/articles/CBMiwAFBVV95cUxQY2RTaTVFenhkSlpaSVFscG5SVF9nR3hHeWhFTzRNMDVkSHdSTzFYclRYSlhNM0lYSmJiQ3RSMzJ3R3dhdmNyck5EcDdGcWtONDFOYjBCSUZNbDZmQ0RzWU1fYmkxT0RSX29QcURiRFg2WFVGazhEaDN2X3dYNE44REhreEhYMEk5QXJxZEpEc3A3eFl2MEhmazNPVWd0U1lKWXZ3dnFVcVhOZzIteGgwX0lMV1h3bWJReGlzbUVTeDQ?oc=5',
  'content': 'New Windows 11 build makes mandatory Microsoft Account sign-in even more mandatory - Ars Technica'},
 {'url': 'https://news.google.com/rss/articles/CBMi1AFBVV95cUxNN3NtclpTaTBXeHpkT1hDRkNDRVRCdDZFMVlCTTE1eDNrTXJvT2stcHdHZFRhcFBOUUtvYWRLYXFseFFnc3E3SlIwaUVjbVVnTWpZRlRBamdyVUVLRDRuWXp2R

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from langgraph.prebuilt import create_react_agent